<a href="https://colab.research.google.com/github/YKochura/rl-kpi/blob/main/practice/practice1/Maze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random

In [2]:
class MazeEnv:
  def __init__(self, size=5):
    # Ініціалізуємо лабіринт розміром size x size
    self.size = size
    self.maze = self._generate_maze()
    self.agent_pos = [0, 0]
    self.goal_pos = [size - 1, size - 1]
    self.done = False
    self.actions = ['up', 'down', 'left', 'right']
    self.state_space = size * size  # Кількість станів
    self.action_space = len(self.actions)  # Кількість дій

  def _generate_maze(self):
    # Створюємо лабіринт з перешкодами
    maze = np.zeros((self.size, self.size), dtype=int)
    num_obstacles = int(self.size * self.size * 0.2)
    obstacles = random.sample(range(1, self.size * self.size - 1), num_obstacles)
    for obs in obstacles:
        maze[obs // self.size][obs % self.size] = 1
    return maze

  def reset(self):
    # Повертаємо лабіринт до початкового стану
    self.agent_pos = [0, 0]
    self.done = False
    return self.agent_pos_to_state(self.agent_pos)

  def agent_pos_to_state(self, pos):
    return pos[0] * self.size + pos[1]

  def step(self, action):
    # Рухаємо агента залежно від вибраної дії
    x, y = self.agent_pos
    if action == 0 and x > 0:  # up
      x -= 1
    elif action == 1 and x < self.size - 1:  # down
      x += 1
    elif action == 2 and y > 0:  # left
      y -= 1
    elif action == 3 and y < self.size - 1:  # right
      y += 1

    if self.maze[x][y] == 1:
      # Якщо перешкода, повертаємо на попереднє місце
      return self.agent_pos_to_state(self.agent_pos), -1, False

    self.agent_pos = [x, y]
    if self.agent_pos == self.goal_pos:
      self.done = True
      return self.agent_pos_to_state(self.agent_pos), 10, True

    return self.agent_pos_to_state(self.agent_pos), -0.1, False

  def render(self):
    # Виводимо лабіринт і позицію агента
    maze_copy = np.copy(self.maze)
    x, y = self.agent_pos
    maze_copy[x][y] = 2
    maze_copy[self.goal_pos[0]][self.goal_pos[1]] = 3
    print(maze_copy)

In [4]:
class QLearningAgent:
  def __init__(self, env, alpha=0.1, gamma=0.99, epsilon=0.1):
    self.env = env
    self.q_table = np.zeros((env.state_space, env.action_space))
    self.alpha = alpha  # Швидкість навчання
    self.gamma = gamma  # Коефіцієнт знижки
    self.epsilon = epsilon  # Ймовірність дослідження

  # TODO
  def choose_action(self, state):
    """
    Ця функція реалізовує ε-жадiбну стратегiю для вибору дій агентом

    Параметри:
    state -- поточний стан в якому знаходиться агент (індекс або координата в таблиці станів)

    Повертає:
    Випадкову дію або найкращу дію на основі Q-таблиці
    """

    # BEGIN_YOUR_CODE
    # Генеруємо випадкове число від 0 до 1
    if random.uniform(0, 1) < self.epsilon:
        # Дослідження: обираємо випадкову дію
        action = random.randint(0, self.env.action_space - 1)
    else:
        # Використання досвіду: обираємо найкращу дію за Q-таблицею
        action = np.argmax(self.q_table[state])
    return action
    # END_YOUR_CODE


  def update_q_value(self, state, action, reward, next_state):
    """
    Ця функція оновлює значення Q-функції для поточного стану та вибраної дії на основі отриманої винагороди і майбутніх дій.

    Параметри:
    state -- поточний стан в якому знаходиться агент (індекс або координата в таблиці станів)
    action -- дія, яку агент виконав у цьому стані (індекс дії в таблиці дій)
    reward -- винагорода, яку агент отримав після виконання дії у цьому стані
    next_state -- наступний стан в який перейшов агент після виконання дії

    Повертає:
    Функція нічого не повертає. Вона оновлює внутрішню таблицю self.q_table в рамках агента.
    """

    # BEGIN_YOUR_CODE
    # Знаходимо максимальне значення Q для наступного стану
    best_next_q = np.max(self.q_table[next_state])

    # Застосовуємо формулу оновлення Q-таблиці
    self.q_table[state, action] = self.q_table[state, action] + self.alpha * (reward + self.gamma * best_next_q - self.q_table[state, action])
    # END_YOUR_CODE

  def train(self, episodes):
    for episode in range(episodes):
        state = self.env.reset()
        done = False
        while not done:
            action = self.choose_action(state)
            next_state, reward, done = self.env.step(action)
            self.update_q_value(state, action, reward, next_state)
            state = next_state
        if (episode + 1) % 100 == 0:
            print(f"Episode: {episode + 1}")
            print(f"Q-таблиця: {self.q_table}")

In [5]:
# Створення середовища лабіринту
env = MazeEnv(size=5)
agent = QLearningAgent(env)

In [6]:
# Тренуємо агента
agent.train(episodes=1000)

Episode: 100
Q-таблиця: [[ 0.37218488  6.61837056  0.74827311 -0.254188  ]
 [-0.20715239  0.51635242 -0.2071854  -0.29082936]
 [ 0.          0.          0.          0.        ]
 [-0.1193422  -0.11846325 -0.1999     -0.1211937 ]
 [-0.1193422  -0.12725666 -0.12338904 -0.1271318 ]
 [ 0.64584954 -0.14882035  0.14071762  7.6801    ]
 [-0.1730596  -0.14986687  0.56878451  8.45066143]
 [ 0.96765802  9.00592689 -0.1184929   0.07908878]
 [-0.1143625  -0.11653168  2.06137431 -0.11794924]
 [-0.12597295 -0.11523572 -0.11976408 -0.11845918]
 [-0.16959122 -0.3691291  -0.16759551  0.52204336]
 [-0.13419071 -0.27917813 -0.12912801  4.0328149 ]
 [ 1.11988659  9.34010943  0.43633263  0.18029244]
 [-0.10955834 -0.19614911  2.36064267 -0.10885338]
 [-0.11844348 -0.1999     -0.10916102 -0.10945165]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [-0.06607686  9.58677144  0.75657974  0.83272011]
 [ 0.          0.          0.          0.        ]
 [ 0.  

In [7]:
# Тестуємо агента
state = env.reset()
env.render()

done = False
while not done:
    action = agent.choose_action(state)
    next_state, reward, done = env.step(action)
    state = next_state
    env.render()
    if done:
        print("Агент досяг мету!")

[[2 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 1 1]
 [0 0 0 0 3]]
[[0 0 1 0 0]
 [2 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 1 1]
 [0 0 0 0 3]]
[[0 0 1 0 0]
 [0 2 0 0 0]
 [0 0 0 0 0]
 [1 1 0 1 1]
 [0 0 0 0 3]]
[[0 0 1 0 0]
 [0 0 2 0 0]
 [0 0 0 0 0]
 [1 1 0 1 1]
 [0 0 0 0 3]]
[[0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 2 0 0]
 [1 1 0 1 1]
 [0 0 0 0 3]]
[[0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 2 1 1]
 [0 0 0 0 3]]
[[0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 1 1]
 [0 0 2 0 3]]
[[0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 1 1]
 [0 0 0 2 3]]
[[0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [1 1 0 1 1]
 [0 0 0 0 3]]
Агент досяг мету!
